# T1 — Pipeline reprodutível: `dvc.yaml` e `dvc repro`

**Objetivo:** transformar "rodei uns scripts na ordem certa" em um pipeline declarado,
que só re-executa o que mudou e deixa um recibo (`dvc.lock`) de tudo que foi usado.

```
data/raw/bike.csv
   │
prepare  ─► data/prepared/bike.csv           ✔ já vem pronto
   │
split    ─► data/split/{train,test}.csv      ✔ já vem pronto       (params: split)
   │
train    ─► models/                          👥 Parte 1, juntos     (params: train, mlflow)
   │
evaluate ─► metrics.json                     🧑‍💻 Parte 2, sozinhos
plot     ─► reports/previsto_vs_real.png     🧑‍💻 Parte 2, sozinhos
```

| Chave do estágio | O que declara | Exemplo |
|---|---|---|
| `cmd` | o comando que o estágio roda | `python src/split.py` |
| `deps` | arquivos que ele **lê** (mudou um → roda de novo) | `src/split.py`, `data/prepared` |
| `params` | seções do `params.yaml` que ele usa | `split` |
| `outs` | o que ele **gera** (vai para o cache do DVC) | `data/split` |
| `metrics` | arquivo de métricas (`dvc metrics show/diff`) | `metrics.json` |
| `plots` | gráficos (`dvc plots show`) | `reports/previsto_vs_real.png` |

> 💡 As células `%%bash` rodam comandos de terminal. Se preferir, copie-os para o terminal.
> Os comandos que mudam o Git e o DVC vêm **comentados** (com `#`), para não rodarem sem querer.
> Quando chegar a hora, tire o `#` (selecione as linhas e use **Ctrl + /**) e rode a célula.

In [1]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

---
# Parte 1 · Juntos

## 1. Inicializar o DVC e configurar o remote

O `dvc init` cria a pasta `.dvc/` (configuração e cache). O **remote** é onde o DVC guarda os arquivos que ele
controla: nesta aula, uma **pasta local** (`../dvc-storage`) que simula um storage na nuvem.
Na Aula 2 trocamos por um remote de verdade (DagsHub).

In [2]:
%%bash
# dvc init
# dvc remote add -d local ../dvc-storage
cat .dvc/config
# git add .dvc .dvcignore
# git commit -m "Inicializa DVC com remote local"

Initialized DVC repository.



You can now commit the changes to git.



+---------------------------------------------------------------------+
|                           

                                          |
|        DVC has enabled anonymous aggregate usage analy

tics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|           

  <https://dvc.org/doc/user-guide/analytics>              |
|                                       

... (3 outputs left out)

Setting 'local' as a default remote.


[core]
    remote = local
['remote "local"']
    url = ../../dvc-storage


[main b9786f0] Inicializa DVC com remote local
 3 files changed, 10 insertions(+)
 create mode 10064

4 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvcignore


## 2. Ponto de partida: o pipeline com 2 estágios prontos

Os estágios leem os parâmetros do `params.yaml`:

In [3]:
%%bash
cat params.yaml

split:
  test_days: 28          # últimos 28 dias viram teste

train:
  learning_rate: 0.1
  max_it

er: 300
  max_leaf_nodes: 31
  random_state: 42

mlflow:
  experiment: bike-demand-pipeline


A célula abaixo grava o `dvc.yaml` na raiz do projeto (abra o arquivo no editor para ver).
Leia os dois estágios com calma: **os próximos seguem o mesmo molde.**

In [4]:
%%writefile dvc.yaml
# Pipeline do projeto. Cada estágio declara o comando (cmd), o que ele lê (deps),
# os parâmetros do params.yaml que usa (params) e o que ele gera (outs).
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
    - src/prepare.py
    - src/bike.py
    - data/raw/bike.csv
    outs:
    - data/prepared

  split:
    cmd: python src/split.py
    deps:
    - src/split.py
    - src/bike.py
    - data/prepared
    params:
    - split
    outs:
    - data/split

Writing dvc.yaml


In [5]:
%%bash
dvc dag
# dvc repro

+---------+  
| prepare |  
+---------+  
      *      
      *      
      *      
 +-------+   
 |

 split |   
 +-------+   


Running stage 'prepare':


> python src/prepare.py


prepare: 8,645 linhas (0 removidas) e 14 colunas -> data/prepared/bike.csv


Generating lock file 'dvc.lock'


... (4 outputs left out)

Updating lock file 'dvc.lock'



To track the changes with git, run:

	git add data/.gitignore dvc.lock

To enable auto staging, run

:

	dvc config core.autostage true


Use `dvc push` to send your updates to remote storage.


## 3. Escrever o estágio `train`

Completamos juntos o estágio de treino. A célula regrava o `dvc.yaml` inteiro (os 2 estágios prontos + `train`),
então pode rodá-la de novo sem medo de duplicar nada.

In [6]:
%%writefile dvc.yaml
# Pipeline do projeto. Cada estágio declara o comando (cmd), o que ele lê (deps),
# os parâmetros do params.yaml que usa (params) e o que ele gera (outs).
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
    - src/prepare.py
    - src/bike.py
    - data/raw/bike.csv
    outs:
    - data/prepared

  split:
    cmd: python src/split.py
    deps:
    - src/split.py
    - src/bike.py
    - data/prepared
    params:
    - split
    outs:
    - data/split

  train:
    cmd: python src/train.py
    deps:
    - src/train.py
    - src/bike.py
    - data/split/train.csv
    params:
    - train
    - mlflow
    outs:
    - models

Overwriting dvc.yaml


## 4. Rodar: só o que mudou é executado

`prepare` e `split` não mudaram desde a última execução, então o DVC pula os dois e roda só o `train`.

> 💬 Repare na saída do treino: o `train.py` anota o treino no **MLflow** (experimento `bike-demand-pipeline`)
> e cria uma **versão** do modelo `bike-demand`. Essas duas peças são o assunto da T2 e da T3.

In [7]:
%%bash
dvc dag
# dvc repro

+---------+  
| prepare |  
+---------+  
      *      
      *      
      *      
 +-------+   
 |

 split |   
 +-------+   
      *      
      *      
      *      
 +-------+   
 | train |   
 +--

-----+   


Stage 'prepare' didn't change, skipping


Stage 'split' didn't change, skipping


Running stage 'train':


... (14 outputs left out)

Updating lock file 'dvc.lock'



To track the changes with git, run:

	git add dvc.lock .gitignore



To enable auto staging, run:

	dvc config core.autostage true


Use `dvc push` to send your updates to remote storage.


## 5. Guardar o recibo

O `dvc.lock` é o **recibo**: o hash de cada dependência, parâmetro e saída. Ele vai para o Git; as saídas
(dados preparados, treino/teste e modelo) vão para o remote com `dvc push`.

In [8]:
%%bash
head -20 dvc.lock
# git add dvc.yaml dvc.lock .gitignore data/.gitignore
# git commit -m "Pipeline DVC: prepare -> split -> train"
# dvc push

schema: '2.0'
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
    - path: data/raw/bike.

csv
      hash: md5
      md5: fc7d423afbce5699a61ac1a8acc533b8
      size: 562742
    - path: src/b

ike.py
      hash: md5
      md5: c2d8efdd8a57a65e92810794ba981855
      size: 1911
    - path: src/

prepare.py
      hash: md5
      md5: 73e959584caa6e71c287c05ad42dba6f
      size: 768
    outs:
   

 - path: data/prepared
      hash: md5


[main 93d510a] Pipeline DVC: prepare -> split -> train
 4 files changed, 116 insertions(+)
 create m

ode 100644 data/.gitignore
 create mode 100644 dvc.lock
 create mode 100644 dvc.yaml


8 files pushed


## 6. Mudar um parâmetro e voltar atrás

Troque `max_leaf_nodes` de 31 para 63 no `params.yaml` (a célula faz isso por você). O DVC percebe que
só o `train` depende desse parâmetro.

In [9]:
%%bash
# sed -i 's/max_leaf_nodes: 31/max_leaf_nodes: 63/' params.yaml
dvc params diff
# dvc repro

Path         Param                 HEAD    workspace
params.yaml  train.max_leaf_nodes  31      63


Stage 'prepare' didn't change, skipping


Stage 'split' didn't change, skipping


Running stage 'train':


> python src/train.py


2026/11/04 21:25:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle 

... (10 outputs left out)

Updating lock file 'dvc.lock'



To track the changes with git, run:

	git add dvc.lock



To enable auto staging, run:

	dvc config core.autostage true


Use `dvc push` to send your updates to remote storage.


Agora desfaça. O DVC tem um *run-cache*: ele reconhece a combinação antiga e **restaura** o modelo sem treinar de novo.

In [10]:
%%bash
# git checkout params.yaml
# dvc repro

Updated 1 path from the index


Stage 'prepare' didn't change, skipping


Stage 'split' didn't change, skipping


Stage 'train' is cached - skipping run, checking out outputs


Updating lock file 'dvc.lock'



To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config c

ore.autostage true

Use `dvc push` to send your updates to remote storage.


---
# Parte 2 · Sozinhos

## 1. Escreva os estágios `evaluate` e `plot`

Mesmo molde do `train`, com duas chaves novas: `metrics` e `plots`. Complete `cmd` e `deps` de cada estágio.

| Estágio | Script | Lê | Gera |
|---|---|---|---|
| `evaluate` | `src/evaluate.py` | modelo + teste | `metrics.json` (e anexa as métricas ao run do MLflow) |
| `plot` | `src/plot.py` | modelo + teste | `reports/previsto_vs_real.png` (e anexa a figura ao run) |

In [11]:
%%writefile dvc.yaml
# Pipeline do projeto. Cada estágio declara o comando (cmd), o que ele lê (deps),
# os parâmetros do params.yaml que usa (params) e o que ele gera (outs).
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
    - src/prepare.py
    - src/bike.py
    - data/raw/bike.csv
    outs:
    - data/prepared

  split:
    cmd: python src/split.py
    deps:
    - src/split.py
    - src/bike.py
    - data/prepared
    params:
    - split
    outs:
    - data/split

  train:
    cmd: python src/train.py
    deps:
    - src/train.py
    - src/bike.py
    - data/split/train.csv
    params:
    - train
    - mlflow
    outs:
    - models

  evaluate:
    cmd: python src/evaluate.py
    deps:
    - src/evaluate.py
    - src/bike.py
    - models
    - data/split/test.csv
    metrics:
    - metrics.json:
        cache: false

  plot:
    cmd: python src/plot.py
    deps:
    - src/plot.py
    - models
    - data/split/test.csv
    plots:
    - reports/previsto_vs_real.png:
        cache: false

Overwriting dvc.yaml


## 2. Rode o pipeline e veja as métricas (como no passo 4 da Parte 1)

In [12]:
%%bash
dvc dag
# dvc repro
dvc metrics show

            +---------+              
            | prepare |              
            +---------+ 

             
                  *                  
                  *                  
          

        *                  
             +-------+               
             | split |            

   
           **+-------+***            
        ***       *      ***         
     ***          * 

        ***      
  ***             *            ***   
**           +-------+            ** 
 **   

       | train |           **  
   *         +-------+          *    
    *        *        *       

... (13 outputs left out)


To track the changes with git, run:

	git add dvc.lock



To enable auto staging, run:

	dvc config core.autostage true


Use `dvc push` to send your updates to remote storage.


Path          mae      r2       rmse
metrics.json  34.9424  0.72296  54.74162


➡️ Abra `reports/previsto_vs_real.png` no editor: é o real × previsto do modelo no período de teste.

## 3. Guarde o recibo (como no passo 5 da Parte 1)

In [13]:
%%bash
# git add dvc.yaml dvc.lock metrics.json reports/previsto_vs_real.png
# git commit -m "Pipeline DVC: + evaluate e plot"
# dvc push

[main 377d758] Pipeline DVC: + evaluate e plot
 4 files changed, 72 insertions(+)
 create mode 10064

4 metrics.json
 create mode 100644 reports/previsto_vs_real.png


Everything is up to date.


## 4. Um experimento de verdade (como no passo 6 da Parte 1)

No `params.yaml`, mude `max_leaf_nodes` para `15`, ou seja, árvores menores (a célula faz isso por você;
depois, se quiser, teste a sua própria combinação). Rode o pipeline e compare as métricas com o último commit
(`dvc metrics diff`). **Melhorou?** Faça commit. **Piorou?** Desfaça, como no passo 6 da Parte 1.

In [14]:
%%bash
# sed -i 's/max_leaf_nodes: 31/max_leaf_nodes: 15/' params.yaml
dvc params diff
# dvc repro
dvc metrics diff

Path         Param                 HEAD    workspace
params.yaml  train.max_leaf_nodes  31      15


Stage 'prepare' didn't change, skipping


Stage 'split' didn't change, skipping


Running stage 'train':


> python src/train.py


2026/11/04 21:25:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle 

... (21 outputs left out)

Use `dvc push` to send your updates to remote storage.


Path          Metric    HEAD      workspace    Change
metrics.json  mae       34.9424   33.93142    

 -1.01098
metrics.json  r2        0.72296   0.76221      0.03926
metrics.json  rmse      54.74162  5

0.71527     -4.02635


In [15]:
%%bash
# git add params.yaml dvc.lock metrics.json reports/previsto_vs_real.png
# git commit -m "HGB: max_leaf_nodes 15"
# dvc push
# git push

[main 8b071f5] HGB: max_leaf_nodes 15
 4 files changed, 14 insertions(+), 14 deletions(-)


3 files pushed

To https://github.com/aluno/mlops-facens.git
   c849714..8b071f5  main -> main


---
## Para discutir
1. No passo 2 da Parte 2, por que só `evaluate` e `plot` rodaram?
2. Para que serve o `dvc.lock`? E o *run-cache*?
3. O `dvc.lock` guarda o md5 do `data/raw/bike.csv` (veja abaixo). Se alguém apagar esse arquivo, o DVC consegue trazê-lo de volta?

In [16]:
%%bash
grep -A3 "path: data/raw/bike.csv" dvc.lock

    - path: data/raw/bike.csv
      hash: md5
      md5: fc7d423afbce5699a61ac1a8acc533b8
      size

: 562742


**Respostas esperadas:**
1. Os estágios anteriores não tiveram nenhuma dependência, parâmetro ou saída alterada. O DVC compara os hashes com o `dvc.lock` e pula o que já está em dia.
2. O `dvc.lock` registra os hashes exatos do que foi usado e gerado: quem clona o repositório reproduz o mesmo resultado.
   O *run-cache* lembra execuções anteriores e restaura as saídas sem rodar de novo.
3. Não. O `dvc.lock` só guarda o hash, para perceber quando o dado muda. O `bike.csv` é uma **dependência** do pipeline,
   não uma saída: não foi para o cache nem para o remote. Versionar o próprio dado (`dvc add`) é o assunto da T4.